# TAD importance genomic validation

In [ ]:
mode = "subset"


In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

colors = [
    "#70acc0",
    "#c12075",
    "#003754",
    "#4c626c",
    "#be9e7c",
    "#723c5d",
    "#9d9c9c",
    "#00695c",
    "#e57373",
]
colors_name = [
    "graublau",
    "purpur",
    "blau",
    "dunkelgrau",
    "bronze",
    "violett",
    "hellgrau",
    "gedämpftes teal",
    "sanftes koralle",
]
sns.set_palette(sns.color_palette(colors))
base_color = "#70acc0"
light_palette = sns.light_palette(base_color, as_cmap=False)
colors = light_palette + ["#4c626c", "#003754"]
cmap = LinearSegmentedColormap.from_list("custom_cmap", colors)
colors_heat = ["#c12075", "#f0f0f0", "#003754"]
cmap_heat = LinearSegmentedColormap.from_list("custom_diverging", colors_heat, N=256)
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"

import re as _re
import matplotlib.figure as _mpl_figure

def _fix_svg_font_declarations(_svg_path):
    """Convert matplotlib's unsupported CSS 'font:' shorthand (labeled px, but
    actually point-sized, in a pt-calibrated viewBox) into explicit
    font-family/font-size/font-weight/font-style, which is what Figma's SVG
    importer requires to resolve size and style correctly."""
    try:
        with open(_svg_path, "r", encoding="utf-8") as _f:
            _content = _f.read()
    except OSError:
        return
    def _convert(_m):
        _oblique = _m.group(1)
        _weight = _m.group(2)
        _pt = float(_m.group(3)) * 0.75
        _family = _m.group(4)
        _font_weight = "bold" if _weight else "normal"
        _font_style = "oblique" if _oblique else "normal"
        return (
            f"font-family: {_family}; font-size: {_pt:g}pt; "
            f"font-weight: {_font_weight}; font-style: {_font_style}"
        )
    _pattern = _re.compile(r'font:\s*(oblique\s+)?(?:(\d+)\s+)?([0-9.]+)px\s+([^;"]+)')
    _new_content = _pattern.sub(_convert, _content)
    if _new_content != _content:
        with open(_svg_path, "w", encoding="utf-8") as _f:
            _f.write(_new_content)

_orig_figure_savefig = _mpl_figure.Figure.savefig

def _rasterize_dense_elements(_fig):
    for _ax in _fig.get_axes():
        for _coll in list(_ax.collections):
            try:
                _coll.set_rasterized(True)
            except Exception:
                pass
        for _im in _ax.get_images():
            try:
                _im.set_rasterized(True)
            except Exception:
                pass

def _patched_figure_savefig(self, fname, *args, **kwargs):
    if isinstance(fname, str) and fname.lower().endswith(".svg"):
        _rasterize_dense_elements(self)
        kwargs.setdefault("dpi", 300)
    _result = _orig_figure_savefig(self, fname, *args, **kwargs)
    if isinstance(fname, str) and fname.lower().endswith(".svg"):
        _fix_svg_font_declarations(fname)
    return _result

_mpl_figure.Figure.savefig = _patched_figure_savefig
mpl.rcParams["figure.dpi"] = 500.0
mpl.rcParams["savefig.transparent"] = True
mpl.rcParams["savefig.format"] = "png"

mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["DejaVu Sans"]

mpl.rcParams["axes.edgecolor"] = "black"
mpl.rcParams["axes.linewidth"] = 0.8
mpl.rcParams["axes.grid"] = True
mpl.rcParams["grid.color"] = "grey"
mpl.rcParams["grid.linestyle"] = "--"
mpl.rcParams["grid.linewidth"] = 0.5

mpl.rcParams["legend.frameon"] = False
mpl.rcParams["legend.loc"] = "best"

mpl.rcParams["xtick.major.size"] = 3
mpl.rcParams["xtick.minor.size"] = 2
mpl.rcParams["ytick.major.size"] = 3
mpl.rcParams["ytick.minor.size"] = 2
mpl.rcParams["xtick.major.width"] = 0.4
mpl.rcParams["xtick.minor.width"] = 0.3
mpl.rcParams["ytick.major.width"] = 0.4
mpl.rcParams["ytick.minor.width"] = 0.3
scaling_factor = 1
if mode == "all":
    base_font_size = 4
else:
    base_font_size = 6
base_axes_title_size = base_font_size + 1
base_axes_label_size = base_font_size
base_tick_label_size = base_font_size - 0.5
base_legend_font_size = base_font_size - 0.5
base_figure_title_size = base_font_size + 2
mpl.rcParams["font.size"] = base_font_size * scaling_factor
mpl.rcParams["axes.titlesize"] = base_axes_title_size * scaling_factor
mpl.rcParams["axes.labelsize"] = base_axes_label_size * scaling_factor
mpl.rcParams["xtick.labelsize"] = base_tick_label_size * scaling_factor
mpl.rcParams["ytick.labelsize"] = base_tick_label_size * scaling_factor
mpl.rcParams["legend.fontsize"] = base_legend_font_size * scaling_factor
mpl.rcParams["figure.titlesize"] = base_figure_title_size * scaling_factor

mpl.rcParams["patch.linewidth"] = 0.7

mpl.rcParams["figure.autolayout"] = True


In [ ]:
def set_figure_params(column, height):

    single_column_width = 3.5
    double_column_width = 7.0

    if column == "single":
        figure_width = single_column_width
    elif column == "double":
        figure_width = double_column_width
    else:
        figure_width = column
    scaling_factor = figure_width / single_column_width
    scaling_factor = 1

    mpl.rcParams["figure.figsize"] = (figure_width, height)


In [ ]:
import glob
import os
import pickle
import re
import warnings

import numpy as np
import pandas as pd
import scipy.stats
from IPython.display import display
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.patches import Rectangle
from scipy.stats import wilcoxon

warnings.filterwarnings("ignore")
_OUTDIR_ABS = os.path.join(os.getcwd(), "outputs", "figS19_S23_neurips2021_cite_bmmc_fc_interaction_gene_layer_genomic_validation")
os.makedirs(_OUTDIR_ABS, exist_ok=True)
PRIMARY = "../.experiment_data/primary"
CHIP = "../.experiment_data/reruns/fig4_tad/chip"
RESOURCES = "../preprocessing/resources"
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 120)

## Loading

In [ ]:
SEEDS = [11, 13, 14]
AUROC, HELDOUT = "predictors_label_valid_roc", "predictors_label_test_roc"
FOLDER = {"go": "calc_go_terms_cell_type", "genes": "calc_genes_cell_type", "groups": "calc_gene_groups_cell_type"}

networkvi_tad_paths = {
    11: "../.experiment_data/reruns/fig4_tad/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s11/default",
    13: "../.experiment_data/reruns/fig4_tad/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s13/default",
    14: "../.experiment_data/reruns/fig4_tad/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s14/default",
}

networkvi_no_tad_paths = {
    11: "../.experiment_data/reruns/fig4_tad/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_with_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s11/default",
    13: "../.experiment_data/reruns/fig4_tad/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_with_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s13/default",
    14: "../.experiment_data/reruns/fig4_tad/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_with_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s14/default",
}


def run_dir(arch, seed, tad_map="ENCFF173VDJ"):
    if arch == "notad":
        return networkvi_no_tad_paths[seed]
    if tad_map == "ENCFF173VDJ":
        return networkvi_tad_paths[seed]
    raise KeyError(tad_map)


def cell_type_key(label):
    return label.replace("/", "_").replace(" ", "_")


def nice(ct):
    return ct.replace("_", " ").replace("G M prog", "G/M prog").replace("MK E prog", "MK/E prog")


def load_importances(directory, kind, modality="expression"):
    out = {}
    for f in sorted(glob.glob(f"{directory}/{FOLDER[kind]}/{modality}_cell_type_vs_rest_*_evaluated_graph_csv_with_ci.pkl")):
        label = os.path.basename(f).split("_vs_rest_", 1)[1].split("_res_unp_act_mask")[0]
        out[cell_type_key(label)] = pd.read_pickle(f).apply(pd.to_numeric, errors="coerce")
    return out


def paired(tad, base, column=AUROC):
    return {ct: pd.concat([tad[ct][column].rename("tad"), base[ct][column].rename("base")], axis=1, join="inner") for ct in sorted(set(tad) & set(base))}


def pct(x):
    return 100 * (np.asarray(x) - 1)


def boot_ci(x):
    r = scipy.stats.bootstrap((np.asarray(x, dtype=float),), np.mean, n_resamples=10000, confidence_level=0.95, random_state=0)
    return r.confidence_interval.low, r.confidence_interval.high


def averaged(tables):
    cts = sorted(set.intersection(*[set(t) for t in tables]))
    return {ct: pd.concat([t[ct][AUROC] for t in tables], axis=1, join="inner").mean(axis=1).rename(AUROC).to_frame() for ct in cts}


def gain_ci(ratios):
    lo, hi = boot_ci(ratios)
    return {"value (%)": float(pct(np.mean(ratios))), "95% CI low (%)": float(pct(lo)), "95% CI high (%)": float(pct(hi)), "Wilcoxon p": wilcoxon(ratios, np.ones_like(ratios)).pvalue}

In [ ]:
IMP = {(arch, s): load_importances(run_dir(arch, s), "groups") for arch in ["tad", "notad"] for s in SEEDS}
PAIRS = {("groups", s): paired(IMP[("tad", s)], IMP[("notad", s)]) for s in SEEDS}

In [ ]:
DIFFS = {}
for s in SEEDS:
    p = PAIRS[("groups", s)]
    common = sorted(set.intersection(*[set(x.index) for x in p.values()]))
    DIFFS[s] = pd.DataFrame({ct: (x.tad - x.base).reindex(common) for ct, x in p.items()})
common = sorted(set.intersection(*[set(d.index) for d in DIFFS.values()]))
cell_keys = sorted(set.intersection(*[set(d.columns) for d in DIFFS.values()]), key=nice)
D = sum(d.reindex(index=common, columns=cell_keys) for d in DIFFS.values()) / len(SEEDS)
tad_ids_ordered = list(D.index)
phenotypes_labels = [nice(c) for c in cell_keys]
mean_gain_per_tad = D.mean(axis=1).values
print(D.shape)

In [ ]:
names = pd.read_csv(f"{RESOURCES}/ensembl_gene_id_gene_name_mapping/ensembl_gene_id_gene_name_mapping.csv")
gene_name = {g: n for g, n in zip(names["Gene stable ID"], names["Gene name"]) if isinstance(n, str) and n}
name = lambda g: gene_name.get(g, g)
tad_to_label = lambda t: ", ".join(name(g) for g in t.split("_"))
positions = pd.read_csv(f"{RESOURCES}/ensembl_gene_id_chromosome_position_mapping/ensembl_gene_id_chromosome_position_mapping.csv")


def tad_loci(tad_ids):
    pos = positions.groupby("Gene stable ID").agg({"Chromosome/scaffold name": "first", "Gene start (bp)": list, "Gene end (bp)": list})
    out = {}
    for t in tad_ids:
        genes = [g for g in t.split("_") if g in pos.index]
        if not genes:
            continue
        starts = [x for g in genes for x in pos.loc[g, "Gene start (bp)"]]
        ends = [x for g in genes for x in pos.loc[g, "Gene end (bp)"]]
        out[t] = ("chr" + str(pos.loc[genes[0], "Chromosome/scaffold name"]), min(starts), max(starts), max(ends))
    return out

## CpG islands

In [ ]:
cpg = pd.read_csv(f"{PRIMARY}/cpg_islands_hg38.bed", sep="\t", header=None, names=["chrom", "start", "end"])
cpg_by_chrom = {c: d for c, d in cpg.groupby("chrom")}
density = {}
for t, (chrom, start, _, end) in tad_loci(tad_ids_ordered).items():
    if not (chrom[3:].isdigit() or chrom in ("chrX", "chrY")):
        continue
    d = cpg_by_chrom.get(chrom)
    density[t] = (0 if d is None else int(((d.start < end) & (d.end > start)).sum())) / max(1, (end - start) / 1000)
cpg_density = pd.Series(density, dtype=float)
x = cpg_density.reindex(tad_ids_ordered).values.astype(float)
ok = ~np.isnan(x)
set_figure_params("single", 2.2)
fig, ax = plt.subplots()
ax.scatter(x[ok], mean_gain_per_tad[ok], alpha=0.4, s=3, color="#70acc0")
ax.set_xlabel("CpG island density (islands per kb)")
ax.set_ylabel("Mean TAD importance gain")
rho_cpg = scipy.stats.spearmanr(x[ok], mean_gain_per_tad[ok])
ax.set_title(f"Spearman ρ = {rho_cpg.statistic:.3f}, p = {rho_cpg.pvalue:.3e}")
plt.tight_layout()
plt.savefig(os.path.join(_OUTDIR_ABS, "cpg_density_vs_tad_importance_gain.pdf"), bbox_inches="tight")
plt.savefig(os.path.join(_OUTDIR_ABS, "cpg_density_vs_tad_importance_gain.svg"), bbox_inches="tight", format="svg")
plt.savefig(os.path.join(_OUTDIR_ABS, "cpg_density_vs_tad_importance_gain.png"), bbox_inches="tight", dpi=500)
plt.show()

## ChIP-seq correlation

In [ ]:
per_track = pd.concat([pd.read_csv(f, sep="\t") for f in sorted(glob.glob(f"{CHIP}/per_track/*.tsv"))])
signal = per_track.pivot_table(index="tad", columns="accession", values="max")
target = per_track.drop_duplicates("accession").set_index("accession")["target"]
lab = pd.read_csv(f"{CHIP}/fc_track_selection.tsv", sep="\t", header=None).set_index(5)[2]


def corr_tracks(sig, imp):
    ok = ~np.isnan(imp)
    out = []
    for j in range(sig.shape[1]):
        m = ok & ~np.isnan(sig[:, j])
        out.append(scipy.stats.spearmanr(sig[m, j], imp[m]).statistic if m.sum() >= 5 else np.nan)
    return np.array(out)


avg = {arch: {ct: pd.concat([IMP[(arch, s)][ct][AUROC] for s in SEEDS], axis=1, join="inner").mean(axis=1) for ct in sorted(set.intersection(*[set(IMP[(arch, s)]) for s in SEEDS]))} for arch in ["tad", "notad"]}
cts = sorted(set(avg["tad"]) & set(avg["notad"]))
tads = sorted(set(avg["tad"][cts[0]].index) & set(avg["notad"][cts[0]].index) & set(signal.index))
sig = signal.loc[tads].values
chip = pd.concat([pd.DataFrame({"cell_type": ct, "accession": signal.columns, "rho_tad": corr_tracks(sig, avg["tad"][ct].reindex(tads).values), "rho_base": corr_tracks(sig, avg["notad"][ct].reindex(tads).values)}) for ct in cts], ignore_index=True)
chip["target"], chip["lab"] = chip.accession.map(target), chip.accession.map(lab)
chip["delta"] = chip.rho_tad - chip.rho_base
factors = chip.groupby(["accession", "target", "lab"]).agg(rho_tad=("rho_tad", "mean"), rho_base=("rho_base", "mean"), delta=("delta", "mean")).reset_index()
factors["rank"] = factors.delta.rank(ascending=False).astype(int)
factors["p_cell_types"] = [scipy.stats.wilcoxon(chip[chip.accession == acc].dropna().pipe(lambda x: x.rho_tad - x.rho_base)).pvalue for acc in factors.accession]
factors = factors.sort_values("rank").reset_index(drop=True)
print(len(factors), "experiments;", int((factors.delta > 0).sum()), "with a positive correlation gain; mean rho full model", round(chip.rho_tad.mean(), 3), "ablated", round(chip.rho_base.mean(), 3))
display(factors.head(20).round(4))
display(factors[factors.target.isin(["CTCF", "RAD21", "SMC3"])].round(4))

In [ ]:
from matplotlib.colors import TwoSlopeNorm

def get_custom_cmap(vmin, vmax):
    if vmin < 0:
        colors_heat = ['#c12075', '#f0f0f0', '#003754']
    else:
        colors_heat = ['#f0f0f0', '#003754']
    cmap_heat = LinearSegmentedColormap.from_list('custom_diverging', colors_heat, N=256)
    return cmap_heat

def plot_heatmap_gene_interactions_layer(features_matrix, xlabels, ylabels, part_indices, add_marker_labels, mode, figsize=(10, 10), vmin=None, vmax=None, row_linkage=None, column_linkage=None, save_name=False):
    features_matrix_mask = np.zeros_like(features_matrix)
    for n_part_index, part_index in enumerate(part_indices):
        features_matrix = np.insert(features_matrix, part_index + n_part_index * 1, np.zeros((1, features_matrix.shape[0])), axis=1)
        features_matrix_mask = np.insert(features_matrix_mask, part_index + n_part_index * 1, np.ones((1, features_matrix_mask.shape[0])), axis=1)
        xlabels = xlabels[:part_index + n_part_index * 1] + [''] + xlabels[part_index + n_part_index * 1:]
    if vmin is None:
        vmin = min(0, np.min(features_matrix))
    if vmax is None:
        vmax = np.max(features_matrix)
    midpoint = 0
    if vmin < 0:
        norm = TwoSlopeNorm(vmin=vmin, vcenter=midpoint, vmax=vmax)
        cmap_heat = get_custom_cmap(vmin, vmax)
        if row_linkage is None and column_linkage is None:
            g = sns.clustermap(features_matrix, figsize=(7, 4), norm=norm, mask=features_matrix_mask, xticklabels=xlabels, yticklabels=ylabels if add_marker_labels is True else [], cmap=cmap_heat, vmin=vmin, vmax=vmax, col_cluster=True, cbar_pos=(1.05, 0.4, 0.03, 0.2))
        else:
            g = sns.clustermap(features_matrix, figsize=(7, 4), norm=norm, mask=features_matrix_mask, xticklabels=xlabels, yticklabels=ylabels if add_marker_labels is True else [], cmap=cmap_heat, vmin=vmin, vmax=vmax, col_cluster=True, row_linkage=row_linkage, col_linkage=column_linkage, cbar_pos=(1.05, 0.4, 0.03, 0.2))
    else:
        cmap_heat = get_custom_cmap(vmin, vmax)
        if row_linkage is None and column_linkage is None:
            g = sns.clustermap(features_matrix, figsize=(7, 4), mask=features_matrix_mask, xticklabels=xlabels, yticklabels=ylabels if add_marker_labels is True else [], cmap=cmap_heat, vmin=vmin, vmax=vmax, col_cluster=True, cbar_pos=(1.0, 0.4, 0.03, 0.2))
        else:
            g = sns.clustermap(features_matrix, figsize=(7, 4), mask=features_matrix_mask, xticklabels=xlabels, yticklabels=ylabels if add_marker_labels is True else [], cmap=cmap_heat, vmin=vmin, vmax=vmax, col_cluster=True, row_linkage=row_linkage, col_linkage=column_linkage, cbar_pos=(1.0, 0.4, 0.03, 0.2))
    row_linkage = g.dendrogram_row.linkage
    column_linkage = g.dendrogram_col.linkage
    plt.xticks(rotation=90)
    g.ax_heatmap.tick_params(axis='x')
    g.ax_heatmap.tick_params(axis='y')
    if add_marker_labels is True:
        g.ax_heatmap.tick_params(axis='y')
    else:
        g.ax_heatmap.tick_params(axis='y', which='both', left=False, right=False, labelleft=False)
        g.ax_heatmap.set_ylabel('genomic markers')
    if save_name:
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.png'), dpi=500, format='png', transparent=True, bbox_inches='tight')
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.svg'), dpi=500, transparent=True, bbox_inches='tight', format='svg')
    else:
        plt.show()
    return (row_linkage, column_linkage)


In [ ]:
cm = chip.groupby(["accession", "cell_type"])[["rho_tad", "delta"]].mean().reset_index()
meta = chip.drop_duplicates("accession").set_index("accession")[["target", "lab"]]
M = cm.pivot_table(index="accession", columns="cell_type", values="rho_tad")[cell_keys]
C = cm.pivot_table(index="accession", columns="cell_type", values="delta")[cell_keys]
genomic_markers = list(meta.loc[M.index, "target"])
labs = [str(l).split(",")[0].strip().lower() for l in meta.loc[M.index, "lab"]]
lab_pos = [i for i, l in enumerate(labs) if l == "michael snyder"]
Mv, Cv = M.values, C.values
top_corr = set(np.where(np.abs(Mv) > np.quantile(np.abs(Mv[lab_pos]), 0.999))[0]) & set(lab_pos)
top_change = set(np.where(np.abs(Cv) > np.quantile(np.abs(Cv[lab_pos]), 0.999))[0]) & set(lab_pos)
first = {}
for i in sorted(top_corr | top_change, key=lambda i: (genomic_markers[i], -np.nanmax(np.abs(Mv[i])))):
    first.setdefault(genomic_markers[i], i)
rows = [first[k] for k in sorted(first)]
plot_heatmap_gene_interactions_layer(Mv[rows], phenotypes_labels, list(np.array(genomic_markers)[rows]), [], True, "subset", figsize=(20, len(rows) * 0.2),
                                     save_name="neurips2021_bmmc_interaction_gene_layer_top_michael_snyder_expression_with_interaction_layer")
plt.show()
pd.DataFrame({"accession": M.index[rows], "marker": np.array(genomic_markers)[rows]})